<a href="https://colab.research.google.com/github/yorgi544/data_acquisition/blob/main/cuaca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": -8.65,
	"longitude": 115.2167,
	"hourly": ["temperature_2m", "cloud_cover", "rain", "soil_temperature_6cm", "wind_speed_80m", "snowfall"],
	"timezone": "Asia/Singapore",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {abs(response.Latitude())}°{'S' if response.Latitude() < 0 else 'N'} {abs(response.Longitude())}°{'W' if response.Longitude() < 0 else 'E'}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_soil_temperature_6cm = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_80m = hourly.Variables(4).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(5).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["rain"] = hourly_rain
hourly_data["soil_temperature_6cm"] = hourly_soil_temperature_6cm
hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
hourly_data["snowfall"] = hourly_snowfall

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 8.68189811706543°S 115.19379425048828°E
Elevation: 35.0 m asl
Timezone: b'Asia/Singapore'b'GMT+8'
Timezone difference to GMT+0: 28800s

Hourly data
                          date  temperature_2m  cloud_cover  rain  \
0   2026-11-10 00:00:00+08:00       25.350000         30.0   0.0   
1   2026-11-10 01:00:00+08:00       25.250000         70.0   0.0   
2   2026-11-10 02:00:00+08:00       25.100000         84.0   0.0   
3   2026-11-10 03:00:00+08:00       24.900000         84.0   0.0   
4   2026-11-10 04:00:00+08:00       24.850000         83.0   0.0   
..                        ...             ...          ...   ...   
163 2026-11-16 19:00:00+08:00       27.700001         57.0   0.1   
164 2026-11-16 20:00:00+08:00       27.100000         59.0   0.1   
165 2026-11-16 21:00:00+08:00       26.600000         61.0   0.0   
166 2026-11-16 22:00:00+08:00       26.100000         62.0   0.0   
167 2026-11-16 23:00:00+08:00       25.650000         63.0   0.0   

     soil_temperature